# 实验5：ResNet-50 残差块的网络结构及其实现

本实验基于教材代码示例 6-5 至 6-10，演示如何使用 TensorFlow/Keras 构建 ResNet 残差块（Residual Block）及完整网络。

**实验目标：**
1. 理解残差学习（Residual Learning）的核心思想
2. 实现瓶颈残差块（Bottleneck Residual Block）
3. 构建包含残差块的完整网络并查看结构

**背景：深层网络的退化问题**

在深度学习的发展过程中，研究者们发现了一个反直觉的现象——简单地堆叠更多的卷积层，网络性能并不会持续提升，反而会在深度超过一定阈值后出现退化。何恺明等人在论文《Deep Residual Learning for Image Recognition》（2015）中给出了经典实验证据：在CIFAR-10数据集上，56层普通卷积网络的**训练误差**比20层网络更高。注意，这不是过拟合——如果是过拟合，训练误差应该更低才对。训练误差和测试误差同时升高，说明更深的网络在优化上遇到了根本性的困难。ResNet通过引入残差学习和跳跃连接，成功解决了这一问题，使得训练上百层甚至上千层的网络成为可能。

## 一、导入必要的库

注意新旧写法的区别：
- **旧写法**：`from keras.layers import ...`（适用于独立安装的 Keras 1.x/2.x）
- **新写法**：`from tensorflow.keras.layers import ...`（适用于 TensorFlow >= 2.0）

**为什么要区分新旧写法？** 在 TensorFlow 2.0 之前，Keras 是一个独立的库，可以支持多种后端（TensorFlow、Theano、CNTK）。从 TensorFlow 2.0 开始，Keras 被正式整合为 TensorFlow 的高级 API（`tf.keras`）。使用 `tensorflow.keras` 可以确保与 TensorFlow 版本的完全兼容，避免独立 Keras 与 TensorFlow 之间的版本冲突。

下面代码中我们还导入了 `GlobalAveragePooling2D`（全局平均池化层），它在后续构建完整 ResNet 网络时会用到，将特征图压缩为一维向量。

In [ ]:
# 代码示例：基于教材 6-5 / 6-6
# [旧写法] from keras.layers import Input, Activation, Conv2D, BatchNormalization, add, MaxPooling2D
# [旧写法] from keras.models import Model
# [旧写法] from keras.layers import Dense, Flatten
# [旧写法] from keras.optimizers import Adam

# [新写法] 适用于 TensorFlow >= 2.0
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Activation, Conv2D, BatchNormalization,
    add, MaxPooling2D, Dense, Flatten, GlobalAveragePooling2D
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

print(f"TensorFlow version: {tf.__version__}")

## 二、定义网络参数

设置输入图像尺寸和分类数目。

**参数说明：**
- **NB_CLASS = 3**：分类任务的类别数目。本实验设为3类，实际应用中根据数据集调整（如ImageNet为1000类）。
- **IM_WIDTH = 224, IM_HEIGHT = 224**：输入图像的空间尺寸。224×224是ResNet论文中使用的标准输入尺寸，也是ImageNet数据集的常用预处理尺寸。选择224的原因是经过5次步长为2的下采样后（224→112→56→28→14→7），最终特征图大小恰好为7×7，是一个合适的尺寸。
- **输入通道数为3**：对应RGB三色通道的彩色图像。

In [ ]:
# 代码示例：基于教材 6-6
NB_CLASS = 3       # 分类数目
IM_WIDTH = 224      # 输入图像宽度
IM_HEIGHT = 224     # 输入图像高度

print(f"输入图像尺寸: {IM_WIDTH} x {IM_HEIGHT} x 3")
print(f"分类数目: {NB_CLASS}")

## 三、构建初始卷积层（Stem）

ResNet 的初始部分包含一个 7×7 卷积层（步长为2）和一个最大池化层，用于快速降低空间分辨率。

**空间尺寸变化的详细计算：**

```
输入: 224×224×3
  ↓  Conv2D(64, 7×7, stride=2, padding='same')
     计算: 224 ÷ 2 = 112
112×112×64
  ↓  BatchNormalization + ReLU
112×112×64
  ↓  MaxPooling2D(3×3, stride=2, padding='same')
     计算: 112 ÷ 2 = 56
56×56×64
```

**设计思路解析：**
- **为什么使用7×7大卷积核？** 在网络入口使用较大的卷积核可以在第一层就获得较大的感受野（receptive field），捕获输入图像中较大范围的空间信息（如大的边缘结构）。后续层则使用3×3小卷积核逐步精细提取特征。
- **为什么步长为2而非1？** 步长为2使得输出尺寸减半，快速降低计算量。如果使用步长1，则需要更多的池化层来降维。
- **为什么要保存x0？** `x0 = x` 将池化后的特征图保存下来，作为第一个残差块中跳跃连接的输入。这是实现 H(x) = F(x) + x 的关键——x0 就是公式中的 x，它会跳过主路径的卷积操作，直接与主路径输出相加。

In [ ]:
# 代码示例：基于教材 6-6
# 定义输入层
inpt = Input(shape=(IM_WIDTH, IM_HEIGHT, 3))

# 初始卷积层: 7x7, 64个滤波器, 步长2
x = Conv2D(64, (7, 7), padding='same', strides=(2, 2), activation='relu')(inpt)
x = BatchNormalization()(x)

# 最大池化层: 3x3, 步长2
x = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x)

# 保存当前输出，用于第一个残差块的跳跃连接
x0 = x

print(f"初始卷积层之后的特征图形状: {x.shape}")
print(f"x0 (跳跃连接输入) 形状: {x0.shape}")

## 四、构建一个残差块（Residual Block）

这是 ResNet 的核心部分。一个瓶颈残差块包含三层卷积：

```
输入x (64通道)
  ├─────────────────────────┐
  │                         │ (跳跃连接)
  ▼                         │
Conv2D(64, 1×1) + BN + ReLU │ (降维)
  ▼                         │
Conv2D(64, 3×3) + BN + ReLU │ (提取特征)
  ▼                         │
Conv2D(256, 1×1) + BN       │ (升维)
  ▼                         │
  +  ◄──────────────────────┘ (逐元素相加)
  ▼
  ReLU
  ▼
输出 (256通道)
```

注意：因为输入是64通道，输出是256通道，所以跳跃连接需要用1×1卷积对齐通道数。

**瓶颈结构 vs 标准结构——参数量对比：**

瓶颈结构通过"先降维、再提取、再升维"的策略，大幅减少了参数量和计算量：

| 结构类型 | 各层参数量 | 总参数量 |
|----------|-----------|---------|
| **瓶颈结构** (1×1→3×3→1×1) | 256×64×1×1 + 64×64×3×3 + 64×256×1×1 | **69,632** |
| **标准结构** (3×3→3×3) | 256×256×3×3 + 256×256×3×3 | **1,179,648** |

瓶颈结构的参数量仅为标准结构的约 **5.9%**，却保持了相当的表达能力。这就是 ResNet-50 能够使用50层而不会参数量爆炸的关键设计。

**代码要点预告：**
- 主路径最后一层 1×1 卷积使用 `activation=None`（不激活），因为需要先与跳跃连接相加，再统一做 ReLU
- 跳跃连接使用 1×1 卷积进行通道对齐（64→256），这在论文中称为"投影捷径"（Projection Shortcut）

In [ ]:
# 代码示例：基于教材 6-7
# ========== 残差块主路径 ==========

# 第1层: 1×1卷积 (降维: 64→64)
x = Conv2D(64, (1, 1), padding='same', strides=(1, 1), activation='relu')(x)
x = BatchNormalization()(x)
print(f"1×1卷积(降维)后: {x.shape}")

# 第2层: 3×3卷积 (提取特征)
x = Conv2D(64, (3, 3), padding='same', strides=(1, 1), activation='relu')(x)
x = BatchNormalization()(x)
print(f"3×3卷积(特征提取)后: {x.shape}")

# 第3层: 1×1卷积 (升维: 64→256), 注意这里不使用激活函数
x = Conv2D(256, (1, 1), padding='same', strides=(1, 1), activation=None)(x)
x = BatchNormalization()(x)
print(f"1×1卷积(升维)后: {x.shape}")

# ========== 跳跃连接（Skip Connection） ==========
# 输入通道数(64) != 输出通道数(256)，需要1×1卷积对齐
# 这对应论文中虚线部分的跳跃连接
x0 = Conv2D(256, (1, 1), padding='same', strides=(1, 1), activation='relu')(x0)
x0 = BatchNormalization()(x0)
print(f"跳跃连接(通道对齐)后: {x0.shape}")

# ========== 逐元素相加 + 激活 ==========
x = add([x, x0])          # 将主路径输出与跳跃连接相加
x = Activation('relu')(x)  # 求和后再做一次ReLU
x0 = x                     # 更新x0，作为下一个残差块的输入

print(f"\n残差块输出形状: {x.shape}")

## 五、查看残差块的网络结构

将当前构建的网络封装为 Model，并用 `model.summary()` 查看结构。

**如何阅读 model.summary() 的输出：**
- **Layer (type)**：层的名称和类型（如 conv2d、batch_normalization、add 等）
- **Output Shape**：该层输出张量的形状。格式为 (batch_size, height, width, channels)，其中 `None` 表示 batch_size 在运行时动态确定
- **Param #**：该层可训练参数的数量。例如 Conv2D(64, 3×3) 作用于64通道输入时，参数量 = 64×64×3×3 + 64(bias) = 36,928
- **Connected to**：在函数式API中，显示该层的输入来自哪些层。特别注意 `add` 层会显示两个输入——分别来自主路径和跳跃连接
- **Total params**：模型总参数量。**Non-trainable params** 主要来自 BatchNormalization 层中的均值和方差统计量（这些在推理时使用，训练时通过滑动平均更新，不通过梯度下降优化）

In [ ]:
# 代码示例：基于教材 6-8
# [旧写法] from keras.models import Model
# [新写法] 适用于 TensorFlow >= 2.0
# from tensorflow.keras.models import Model  # 已在上方导入

model = Model(inputs=inpt, outputs=x)
model.summary()

## 六、添加分类头（Classification Head）

在残差块输出之后添加 Flatten 和 Dense 层，用于最终的分类任务。

**为什么需要分类头？** 残差块输出的是三维特征图（高×宽×通道数），而分类任务需要输出一个概率向量（长度等于类别数）。分类头的作用就是将高维特征图转化为分类概率：

1. **Flatten（展平层）**：将三维特征图（如56×56×256 = 802,816维）展平为一维向量。这一步丢弃了空间位置信息，只保留特征值。（注：在完整的ResNet中通常使用GlobalAveragePooling2D代替Flatten，参数量更少且有更好的泛化性能。）
2. **Dense + Softmax（全连接分类层）**：将特征向量映射到类别数维度，Softmax确保输出为概率分布（所有值非负且和为1）。

In [ ]:
# 代码示例：基于教材 6-9
# [旧写法] from keras.layers import Dense, Flatten
# [新写法] 适用于 TensorFlow >= 2.0
# from tensorflow.keras.layers import Dense, Flatten  # 已在上方导入

# 在残差块输出后添加分类层
x = model.output
x = Flatten()(x)
predictions = Dense(NB_CLASS, activation='softmax')(x)

# 构建完整的分类模型
model_res = Model(inputs=model.input, outputs=predictions)
print(f"完整模型输入形状: {model_res.input_shape}")
print(f"完整模型输出形状: {model_res.output_shape}")

## 七、查看完整模型结构

In [ ]:
model_res.summary()

## 八、编译模型

使用 Adam 优化器和交叉熵损失函数编译模型。

注意新旧写法的区别：
- **旧写法**：`Adam(lr=0.001)` 
- **新写法**：`Adam(learning_rate=0.001)`（适用于 TensorFlow >= 2.11）

**编译参数详解：**

- **loss='categorical_crossentropy'（分类交叉熵）**：适用于多分类任务，标签为one-hot编码格式。它衡量的是模型预测的概率分布与真实标签分布之间的差异。交叉熵越小，表示预测越接近真实标签。公式为：$L = -\sum_{i} y_i \log(\hat{y}_i)$，其中 $y_i$ 是真实标签，$\hat{y}_i$ 是预测概率。

- **optimizer=Adam(learning_rate=0.001)（Adam优化器）**：Adam（Adaptive Moment Estimation）结合了Momentum（利用梯度的一阶矩估计加速收敛）和RMSProp（利用梯度的二阶矩估计自适应调整学习率）的优点。learning_rate=0.001是Adam的默认推荐值，在大多数任务上表现良好。

- **metrics=['accuracy']（评估指标）**：在训练和验证过程中，除了损失值外，还同时计算并显示分类准确率，方便直观监控模型性能。

In [ ]:
# 代码示例：基于教材 6-10
# [旧写法] from keras.optimizers import Adam
# [新写法] 适用于 TensorFlow >= 2.0
# from tensorflow.keras.optimizers import Adam  # 已在上方导入

# [旧写法] model_res.compile(loss='categorical_crossentropy', optimizer=Adam(lr=0.001), metrics=['accuracy'])
# [新写法] 适用于 TensorFlow >= 2.11
model_res.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

print("模型编译成功！")
print(f"损失函数: categorical_crossentropy")
print(f"优化器: Adam (learning_rate=0.001)")
print(f"评估指标: accuracy")

## 九、使用函数封装残差块（进阶）

将残差块封装为可复用的函数，方便构建更深的 ResNet 网络。

**函数设计说明：**

`residual_block(x, filters, strides, use_projection)` 函数实现了一个通用的瓶颈残差块：

- **x**：输入张量，即上一层或上一个残差块的输出
- **filters**：瓶颈层（中间层）的滤波器数目。输出通道数自动设为 `filters × 4`（这是ResNet的设计约定，例如 filters=64 时输出256通道，filters=128 时输出512通道）
- **strides**：3×3卷积的步长。默认为 `(1,1)` 表示不下采样；设为 `(2,2)` 时空间尺寸减半，用于不同stage之间的过渡
- **use_projection**：是否在跳跃连接中使用1×1卷积进行维度对齐。需要使用投影的情况有两种：
  1. **通道数不匹配**：如输入64通道但输出256通道（每个stage的第一个块）
  2. **空间尺寸不匹配**：当使用 `strides=(2,2)` 下采样时，主路径的空间尺寸减半，跳跃连接也必须做相应的下采样，否则无法逐元素相加

当 `use_projection=False` 时，跳跃连接直接传递输入（恒等映射），这是残差学习最理想的形式。

In [ ]:
def residual_block(x, filters, strides=(1, 1), use_projection=False):
    """
    构建一个瓶颈残差块（Bottleneck Residual Block）。
    
    参数:
        x: 输入张量
        filters: 瓶颈层的滤波器数目（输出通道数为 filters * 4）
        strides: 3×3卷积的步长，用于下采样
        use_projection: 是否在跳跃连接中使用1×1卷积（通道数不同时需要）
    
    返回:
        输出张量
    """
    shortcut = x  # 保存输入，用于跳跃连接
    
    # 主路径
    # 第1层: 1×1卷积 (降维)
    x = Conv2D(filters, (1, 1), padding='same', strides=(1, 1), activation='relu')(x)
    x = BatchNormalization()(x)
    
    # 第2层: 3×3卷积 (提取特征)
    x = Conv2D(filters, (3, 3), padding='same', strides=strides, activation='relu')(x)
    x = BatchNormalization()(x)
    
    # 第3层: 1×1卷积 (升维)
    x = Conv2D(filters * 4, (1, 1), padding='same', strides=(1, 1), activation=None)(x)
    x = BatchNormalization()(x)
    
    # 跳跃连接
    if use_projection:
        # 通道数不同时，使用1×1卷积对齐
        shortcut = Conv2D(filters * 4, (1, 1), padding='same', strides=strides, activation='relu')(shortcut)
        shortcut = BatchNormalization()(shortcut)
    
    # 逐元素相加 + 激活
    x = add([x, shortcut])
    x = Activation('relu')(x)
    
    return x

print("残差块函数定义完成！")

## 十、使用封装函数构建多残差块网络

使用上面定义的 `residual_block` 函数堆叠多个残差块，构建一个简化版的 ResNet 网络。

**多残差块堆叠的设计逻辑：**

本示例构建了包含 conv2_x 和 conv3_x 两个stage的简化版ResNet（完整的ResNet-50还包含conv4_x和conv5_x）。

**conv2_x阶段（3个残差块，filters=64，输出256通道）：**
- 第1个块：`use_projection=True`，因为输入通道数（64）与输出通道数（256）不匹配，需要1×1卷积对齐
- 第2、3个块：`use_projection=False`（默认），输入输出通道数相同（256→256），跳跃连接为恒等映射
- 空间尺寸保持不变：56×56

**conv3_x阶段（4个残差块，filters=128，输出512通道）：**
- 第1个块：`strides=(2,2)` + `use_projection=True`。这里有两个变化同时发生：
  - **空间下采样**：strides=(2,2) 使得3×3卷积的输出尺寸减半（56×56→28×28），降低后续计算量
  - **通道数变化**：从256通道增加到512通道，跳跃连接需要同时处理空间下采样和通道对齐
- 第2、3、4个块：空间尺寸和通道数保持不变（28×28×512），跳跃连接为恒等映射

**为什么conv3_x第一个块要用strides=(2,2)？** 这是ResNet实现特征层级化的关键机制——每进入一个新的stage，空间分辨率减半、通道数翻倍。这样设计的好处是：更深层的残差块在更小的空间分辨率上操作，感受野相对更大，能够捕获更大范围的语义信息；同时增加通道数以保持表达能力。

In [ ]:
# 构建简化版 ResNet 网络
input_layer = Input(shape=(IM_WIDTH, IM_HEIGHT, 3))

# Stem: 初始卷积 + 池化
x = Conv2D(64, (7, 7), padding='same', strides=(2, 2), activation='relu')(input_layer)
x = BatchNormalization()(x)
x = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x)

# conv2_x: 3个残差块 (filters=64, 输出通道=256)
x = residual_block(x, 64, use_projection=True)   # 第1个块需要通道对齐 (64→256)
x = residual_block(x, 64)                         # 第2个块 (256→256)
x = residual_block(x, 64)                         # 第3个块 (256→256)

# conv3_x: 4个残差块 (filters=128, 输出通道=512, 第1个块下采样)
x = residual_block(x, 128, strides=(2, 2), use_projection=True)  # 下采样 + 通道对齐
x = residual_block(x, 128)
x = residual_block(x, 128)
x = residual_block(x, 128)

# 全局平均池化 + 分类层
x = GlobalAveragePooling2D()(x)
x = Dense(NB_CLASS, activation='softmax')(x)

# 构建模型
resnet_model = Model(inputs=input_layer, outputs=x)
resnet_model.summary()

## 十一、编译多残差块模型

In [ ]:
# [旧写法] resnet_model.compile(loss='categorical_crossentropy', optimizer=Adam(lr=0.001), metrics=['accuracy'])
# [新写法] 适用于 TensorFlow >= 2.11
resnet_model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

print("多残差块 ResNet 模型编译成功！")
print(f"模型总参数量: {resnet_model.count_params():,}")

## 十二、验证模型的前向传播

使用随机数据验证模型能够正确进行前向传播。

In [ ]:
# 生成随机输入数据进行测试
dummy_input = np.random.rand(2, IM_WIDTH, IM_HEIGHT, 3).astype('float32')
dummy_output = resnet_model.predict(dummy_input)

print(f"输入形状: {dummy_input.shape}")
print(f"输出形状: {dummy_output.shape}")
print(f"输出概率 (样本1): {dummy_output[0]}")
print(f"输出概率之和: {dummy_output[0].sum():.6f}  (应接近1.0)")
print(f"预测类别 (样本1): {np.argmax(dummy_output[0])}")

## 新旧写法对照表

| 功能 | 旧写法 | 新写法 |
|------|--------|--------|
| 导入层 | `from keras.layers import add, Conv2D` | `from tensorflow.keras.layers import add, Conv2D` |
| 导入模型 | `from keras.models import Model` | `from tensorflow.keras.models import Model` |
| 学习率 | `Adam(lr=0.001)` | `Adam(learning_rate=0.001)` |
| 训练 | `model.fit_generator(...)` | `model.fit(...)` |

## 思考题

1. 残差学习为什么能解决深层网络的退化问题？
2. 跳跃连接中的1×1卷积有什么作用？什么时候需要？
3. ResNet-50中的"瓶颈"结构（1×1→3×3→1×1）有什么优势？
4. 如果去掉所有的跳跃连接，ResNet还能正常训练吗？
5. BatchNormalization放在ReLU之前还是之后更好？